# RanPAC Phase 2 · Sweep Linear Random Projection đối chứng trực tiếp với Notebook 05

Notebook đối chứng cho [05_ranpac_m_sweep.ipynb](05_ranpac_m_sweep.ipynb): giữ nguyên Split CIFAR-100, class order seed `1993`, ViT-B/16 ImageNet-21K đóng băng (không Phase 1), `FeatureCache` dùng chung, cùng ma trận chiếu Gaussian $W \sim \mathcal{N}(0, 1)$ (seed `1993`), split CV 80/20 và lưới $\lambda \in [10^{-8}, 10^8]$; **chỉ bỏ ReLU**, tức $H = F W$.

### Các mốc $M$ đồng bộ với Notebook 05 + điểm neo $L = 768$
Chạy trên toàn bộ 6 mốc của `SWEEP_DIMS` trong Notebook 05 (`300, 500, 1000, 1250, 2500, 10000`) kèm điểm biên ma trận vuông `768` (`LINEAR_SWEEP_DIMS = (300, 500, 768, 1000, 1250, 2500, 10000)`), đồng thời chạy bổ sung 1 lượt `ReLU` tại `M = 768` (~20 giây) để có **7 cặp điểm đối chứng 1-1 dùng chung ma trận $W$** xuyên suốt hai miền:
- **Miền nén chiều ($M < 768$: `300, 500`):** kiểm tra mất mát thông tin tuyến tính so với việc cắt cụt nửa âm của ReLU và xác định điểm giao cắt (crossover).
- **Điểm biên ($M = L = 768$):** kiểm chứng phép biến đổi tuyến tính khả nghịch và điểm gãy chuyển pha (elbow) của $\operatorname{rank}(G)$.
- **Miền mở rộng chiều ($M > 768$: `1000, 1250, 2500, 10000`):** kiểm chứng Property 2 ($\operatorname{rank}(G_{\text{linear}}) \le 768$, đường accuracy đi ngang trong khi chi phí tính toán tăng theo $M$) và tách hiệu ứng đổi chiều $\Delta_{\text{dim}}(M)$ khỏi đóng góp thuần túy của ReLU $\Delta_{\text{ReLU}}(M)$.


## 1. Cài thư viện và mở mã dùng chung

Đặt `ranpac_core.py`, `ranpac_data.py`, `ranpac_colab.py`, `test_ranpac_core.py` (và `sweep_summary.csv` của Notebook 05 nếu chưa có trong `WORK_DIR/sweeps`) vào `MyDrive/RanPAC_implement`. Giữ nguyên `WORK_DIR` để tái sử dụng cache embedding ViT-B/16. Mọi run log vào project W&B công khai `pnp123/RanPAC-reproduction` với tag `['empirics', 'linear-rp-control']`.


In [ ]:
%pip install -q timm==1.0.22 wandb==0.31.0


In [ ]:
import gc
import sys
import unittest
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from google.colab import drive

drive.mount('/content/drive')
SOURCE_DIR = Path('/content/drive/MyDrive/RanPAC_implement')
WORK_DIR = Path('/content/drive/MyDrive/ranpac-runs')
DATA_ROOT = '/content/ranpac-data'

if not (SOURCE_DIR / 'ranpac_colab.py').is_file():
    raise FileNotFoundError(f'Chưa tìm thấy mã nguồn tại {SOURCE_DIR}')
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

from ranpac_colab import SWEEP_DIMS, configure_wandb, run_phase2, smoke_wandb
from ranpac_core import NUM_TASKS, SEED, gram_spectrum_stats, split_class_order
from ranpac_data import FEATURE_DIM, FeatureCache, atomic_output, feature_config


## 2. Kiểm tra trên CPU và cấu hình Sweep

- Mặc định `MODE = 'sweep'` chạy đầy đủ `LINEAR_SWEEP_DIMS = (300, 500, 768, 1000, 1250, 2500, 10000)` (6 mốc của Notebook 05 + điểm neo `768`, đặt `INCLUDE_M768 = False` nếu chỉ muốn đúng 6 mốc của Notebook 05).
- Bật `RUN_RELU_M768 = True` (mặc định bật, tốn ~20s) để bổ sung điểm `ReLU` tại `M = 768` (vốn chưa có trong Notebook 05), giúp 7 mốc đều có đủ cặp `(Linear, ReLU)` dùng chung ma trận $W$.
- Nếu chỉ muốn chạy thử lẻ 1 cấu hình, đổi `MODE = 'single'` và chọn `SINGLE_PRESET`.


In [ ]:
suite = unittest.defaultTestLoader.loadTestsFromName('test_ranpac_core')
result = unittest.TextTestRunner(verbosity=2).run(suite)
if not result.wasSuccessful():
    raise AssertionError('Kiểm tra CPU chưa đạt')


In [ ]:
WANDB_ENTITY = 'pnp123'
WANDB_PROJECT = 'RanPAC-reproduction'
WANDB_MODE = 'online'
RUN_LABEL = 'linear-sweep-01'
RELU_SWEEP_LABEL = 'm-sweep-01'     # Nhãn run của Notebook 05 trên Drive (nếu có checkpoint để đọc rank/λ)

MODE = 'sweep'                      # 'sweep' (tất cả mốc M) hoặc 'single' (1 mốc đơn lẻ)
INCLUDE_M768 = True                 # Giữ điểm biên M = L = 768 bên cạnh 6 mốc của Notebook 05
RUN_RELU_M768 = True                # Chạy thêm ReLU tại M=768 (~20s) để đủ 7 cặp (Linear, ReLU)

SINGLE_PRESET = 'rp-linear-m768'    # Chỉ dùng khi MODE == 'single'
STOP_AFTER_TASK = 10
RESUME = False                      # Chỉ dùng khi MODE == 'single'

BATCH_SIZE = 48
NUM_WORKERS = 2
STATS_DEVICE = 'cpu'
RIDGE_BASELINE_A10_PCT = 87.02

LINEAR_SWEEP_DIMS = tuple(sorted((*SWEEP_DIMS, *( (768,) if INCLUDE_M768 else () ))))
print('Các mốc M sẽ chạy đối chứng:', LINEAR_SWEEP_DIMS)

RUN_WANDB_SMOKE = False
if RUN_WANDB_SMOKE:
    smoke_wandb(WANDB_ENTITY, WANDB_PROJECT)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Chọn GPU runtime trên Colab trước khi chạy.')
if WANDB_MODE == 'online':
    configure_wandb(WANDB_ENTITY, WANDB_MODE)

feature_device = torch.device('cuda')
cache = FeatureCache(
    WORK_DIR / 'embeddings', DATA_ROOT, feature_config(split_class_order(), feature_device),
    feature_device, BATCH_SIZE, NUM_WORKERS,
)
for task in range(1, NUM_TASKS + 1):
    for split in ('train', 'test'):
        shard = cache.root / f'{split}-task-{task:02}.pt'
        if not shard.is_file():
            print(f'Trích embedding {split} cho task {task:02}...', flush=True)
            cache.load(split, task)
print(f'Cache đã sẵn sàng tại {cache.root}')
del cache
gc.collect()
torch.cuda.empty_cache()


## 3. Thực thi các lượt chạy Linear RP (và điểm ghép cặp ReLU `M=768`)

Ở chế độ `MODE = 'sweep'`, hàm tự kiểm tra từng mốc $M$: nếu đã hoàn tất đủ 10 task thì đọc lại kết quả, nếu bị ngắt giữa chừng thì tự động `resume=True` trên cùng W&B run ID. Sau task 10, cả `gram_rank` chuẩn và chẩn đoán phổ trị riêng `gram_spectrum_stats(G)` đều được lưu lại.


In [ ]:
def ensure_complete(preset, label, *, tags, group):
    run_dir = WORK_DIR / 'runs' / f'{preset}-s{SEED}-{label}'
    ckpt_path = run_dir / 'checkpoint.pt'
    if ckpt_path.is_file():
        saved = torch.load(ckpt_path, map_location='cpu', weights_only=True)
        if len(saved.get('records', ())) == NUM_TASKS:
            print(f'[Đã xong] {run_dir.name} -> A[10] = {saved["records"][-1]["avg_accuracy_pct"]:.2f}%')
            return saved, run_dir
        resume_run = True
    else:
        resume_run = False

    run_phase2(
        preset, WORK_DIR, entity=WANDB_ENTITY, project=WANDB_PROJECT,
        stop_after_task=NUM_TASKS, run_label=label, resume=resume_run,
        wandb_mode=WANDB_MODE, batch_size=BATCH_SIZE, workers=NUM_WORKERS,
        stats_device=STATS_DEVICE, measure_rank=True,
        extra_tags=tags, wandb_group_override=group,
    )
    return torch.load(ckpt_path, map_location='cpu', weights_only=True), run_dir


linear_runs = {}
relu_m768_ckpt = None

if MODE == 'single':
    records = run_phase2(
        SINGLE_PRESET, WORK_DIR, entity=WANDB_ENTITY, project=WANDB_PROJECT,
        stop_after_task=STOP_AFTER_TASK, run_label=RUN_LABEL, resume=RESUME,
        wandb_mode=WANDB_MODE, batch_size=BATCH_SIZE, workers=NUM_WORKERS,
        stats_device=STATS_DEVICE, measure_rank=True,
        extra_tags=['empirics', 'linear-rp-control'],
    )
    if records:
        final = records[-1]
        print({
            'preset': SINGLE_PRESET,
            'A[10] (%)': final['avg_accuracy_pct'] if final['task'] == NUM_TASKS else None,
            'rank(G)': final.get('gram_rank'),
            'rank tolerance': final.get('gram_rank_tolerance'),
            'cumulative ridge solve (s)': final.get('cumulative_ridge_solve_seconds'),
            'peak host RSS (MiB)': max((r.get('host_peak_rss_mib') or 0) for r in records),
        })
elif MODE == 'sweep':
    group_name = f'cifar100-linear-rp-sweep-{RUN_LABEL}'
    for dim in LINEAR_SWEEP_DIMS:
        ckpt, run_dir = ensure_complete(
            f'rp-linear-m{dim}', RUN_LABEL,
            tags=['empirics', 'linear-rp-control'],
            group=group_name,
        )
        linear_runs[dim] = (ckpt, run_dir)

    if INCLUDE_M768 and RUN_RELU_M768:
        relu_m768_ckpt, _ = ensure_complete(
            'rp-act-relu-m768', RUN_LABEL,
            tags=['empirics', 'linear-rp-control', 'relu-m768-anchor'],
            group=group_name,
        )
else:
    raise ValueError("MODE phải là 'sweep' hoặc 'single'.")


## 4. Tổng hợp đối chứng cặp đôi với Notebook 05 và vẽ 4 góc phân tích

Cell dưới kết hợp kết quả Linear RP vừa chạy với dữ liệu ReLU RP từ Notebook 05 (`sweep_summary.csv`, `sweep_rank_g.csv` và checkpoint trên Drive nếu có) để xuất bảng `linear_vs_relu_m_sweep.csv` cùng biểu đồ 4 panel:
1. **Đường cong $A[10]$ theo $M$** qua miền nén chiều ($M < 768$), điểm biên ($M = 768$) và miền mở rộng chiều ($M > 768$).
2. **Phân rã đóng góp trên cùng ma trận $W$:** tách $\Delta_{\text{dim}}(M) = A_{10}^{\text{Linear}}(M) - A_{10}^{\text{Ridge}}$ và $\Delta_{\text{ReLU}}(M) = A_{10}^{\text{ReLU}}(M) - A_{10}^{\text{Linear}}(M)$.
3. **Điểm gãy chuyển pha của $\operatorname{rank}(G)$ tại $L = 768$:** Linear bị khóa ở $\min(M, 768)$, trong khi ReLU (sau khi khử DC spike) tiếp tục tăng theo $M$.
4. **Đối chiếu chi phí thời gian (`fit + eval`)** giữa hai biến thể theo $M$.


In [ ]:
if MODE == 'sweep' and linear_runs:
    # Đọc bảng ReLU từ Notebook 05 (ưu tiên trên Drive, dự phòng trong SOURCE_DIR)
    drive_relu_csv = WORK_DIR / 'sweeps' / f'm-sweep-s{SEED}-{RELU_SWEEP_LABEL}' / 'sweep_summary.csv'
    src_relu_csv = SOURCE_DIR / 'sweep_summary.csv'
    relu_df = pd.read_csv(drive_relu_csv if drive_relu_csv.is_file() else src_relu_csv).set_index('M')

    drive_rank_csv = WORK_DIR / 'sweeps' / f'm-sweep-s{SEED}-{RELU_SWEEP_LABEL}' / 'sweep_rank_g.csv'
    src_rank_csv = SOURCE_DIR / 'sweep_rank_g.csv'
    rank_csv_df = (
        pd.read_csv(drive_rank_csv if drive_rank_csv.is_file() else src_rank_csv).set_index('M')
        if (drive_rank_csv.is_file() or src_rank_csv.is_file()) else pd.DataFrame()
    )

    rows = []
    for dim in LINEAR_SWEEP_DIMS:
        ckpt_lin, _ = linear_runs[dim]
        recs_lin = ckpt_lin['records']
        spec_lin = gram_spectrum_stats(ckpt_lin['model']['G'])
        a10_lin = recs_lin[-1]['avg_accuracy_pct']
        fit_eval_lin = sum(r['fit_seconds'] + r['eval_seconds'] for r in recs_lin)
        solve_lin = sum(r['ridge_solve_seconds'] or 0.0 for r in recs_lin)
        lam10_lin = recs_lin[-1]['ridge_lambda']

        a10_relu = fit_eval_relu = rank_relu_std = rank_relu_deflated = lam10_relu = None
        if dim in relu_df.index:
            a10_relu = float(relu_df.loc[dim, 'a10_pct'])
            fit_eval_relu = float(relu_df.loc[dim, 'fit_eval_seconds'])
        if dim in rank_csv_df.index:
            rank_relu_std = int(rank_csv_df.loc[dim, 'rank_g_relu'])

        # Nếu có checkpoint ReLU của Notebook 05 trên Drive thì tính thêm rank_deflated_dc và λ_10
        relu_ckpt_path = WORK_DIR / 'runs' / f'rp-sweep-m{dim}-s{SEED}-{RELU_SWEEP_LABEL}' / 'checkpoint.pt'
        if dim == 768 and relu_m768_ckpt is not None:
            recs_r = relu_m768_ckpt['records']
            spec_r = gram_spectrum_stats(relu_m768_ckpt['model']['G'])
            a10_relu = recs_r[-1]['avg_accuracy_pct']
            fit_eval_relu = sum(r['fit_seconds'] + r['eval_seconds'] for r in recs_r)
            rank_relu_std = spec_r['rank_standard']
            rank_relu_deflated = spec_r['rank_deflated_dc']
            lam10_relu = recs_r[-1]['ridge_lambda']
        elif relu_ckpt_path.is_file():
            saved_r = torch.load(relu_ckpt_path, map_location='cpu', weights_only=True)
            if saved_r.get('model') is not None and len(saved_r.get('records', ())) == NUM_TASKS:
                spec_r = gram_spectrum_stats(saved_r['model']['G'])
                rank_relu_std = spec_r['rank_standard']
                rank_relu_deflated = spec_r['rank_deflated_dc']
                lam10_relu = saved_r['records'][-1]['ridge_lambda']
            del saved_r

        rows.append({
            'M': dim,
            'a10_linear_pct': a10_lin,
            'a10_relu_pct': a10_relu,
            'delta_dim_vs_ridge_pp': a10_lin - RIDGE_BASELINE_A10_PCT,
            'delta_relu_vs_linear_pp': (a10_relu - a10_lin) if a10_relu is not None else None,
            'rank_g_linear': spec_lin['rank_standard'],
            'rank_g_relu_std_fp32': rank_relu_std,
            'rank_g_relu_deflated_dc': rank_relu_deflated,
            'tol_linear': spec_lin['tol_standard'],
            'lambda_task10_linear': lam10_lin,
            'lambda_task10_relu': lam10_relu,
            'fit_eval_linear_s': fit_eval_lin,
            'fit_eval_relu_s': fit_eval_relu,
            'ridge_solve_linear_s': solve_lin,
            'peak_rss_linear_mib': max((r.get('host_peak_rss_mib') or 0.0) for r in recs_lin),
            'wandb_url_linear': ckpt_lin['wandb'].get('url'),
        })

    df_cmp = pd.DataFrame(rows)
    out_dir = WORK_DIR / 'sweeps' / f'linear-vs-relu-m-sweep-s{SEED}-{RUN_LABEL}'
    with atomic_output(out_dir / 'linear_vs_relu_m_sweep.csv') as tmp:
        df_cmp.to_csv(tmp, index=False)
    display(df_cmp.round(3))

    fig, axes = plt.subplots(2, 2, figsize=(13, 8.5), layout='constrained')

    # Panel 1: A[10] vs M
    ax = axes[0, 0]
    ax.plot(df_cmp['M'], df_cmp['a10_linear_pct'], 's-', label='Linear RP (H = FW)')
    sub_r = df_cmp.dropna(subset=['a10_relu_pct'])
    ax.plot(sub_r['M'], sub_r['a10_relu_pct'], 'o-', label='ReLU RP (H = ReLU(FW))')
    ax.axhline(RIDGE_BASELINE_A10_PCT, linestyle='--', color='gray', label='Ridge không RP (87.02%)')
    ax.axvline(FEATURE_DIM, linestyle=':', color='tab:red', alpha=0.8, label='Ngưỡng L = 768')
    ax.set_ylabel('A[10] (%)')
    ax.set_title('1. Độ chính xác qua miền nén (M<768) & mở rộng (M>768)')
    ax.legend(fontsize=8)

    # Panel 2: Paired decomposition Δ_dim(M) vs Δ_ReLU(M)
    ax = axes[0, 1]
    ax.plot(df_cmp['M'], df_cmp['delta_dim_vs_ridge_pp'], 's--', label='Δ_dim = Linear(M) − Ridge')
    sub_d = df_cmp.dropna(subset=['delta_relu_vs_linear_pp'])
    ax.plot(sub_d['M'], sub_d['delta_relu_vs_linear_pp'], 'o-', label='Δ_ReLU = ReLU(M) − Linear(M) (cùng W)')
    ax.axhline(0.0, linestyle='--', color='gray')
    ax.axvline(FEATURE_DIM, linestyle=':', color='tab:red', alpha=0.8)
    ax.set_ylabel('Chênh lệch A[10] (pp)')
    ax.set_title('2. Phân rã hiệu ứng chiều M và đóng góp thuần của ReLU')
    ax.legend(fontsize=8)

    # Panel 3: Rank phase transition at L = 768
    ax = axes[1, 0]
    ax.plot(df_cmp['M'], df_cmp['rank_g_linear'], 's-', label='rank(G) Linear RP')
    if df_cmp['rank_g_relu_deflated_dc'].notna().any():
        sub_rk = df_cmp.dropna(subset=['rank_g_relu_deflated_dc'])
        ax.plot(sub_rk['M'], sub_rk['rank_g_relu_deflated_dc'], 'o-', label='rank(G) ReLU (deflated DC)')
    if df_cmp['rank_g_relu_std_fp32'].notna().any():
        sub_rs = df_cmp.dropna(subset=['rank_g_relu_std_fp32'])
        ax.plot(sub_rs['M'], sub_rs['rank_g_relu_std_fp32'], '^:', alpha=0.75, label='rank(G) ReLU (std FP32 rtol)')
    ax.plot(df_cmp['M'], df_cmp['M'], 'k:', alpha=0.4, label='Đường chéo full-rank y = M')
    ax.axhline(FEATURE_DIM, linestyle='--', color='tab:red', alpha=0.8, label='Trần tuyến tính L = 768')
    ax.set_yscale('log')
    ax.set_ylabel('Numerical rank(G)')
    ax.set_title('3. Điểm gãy chuyển pha của rank(G) tại M = 768')
    ax.legend(fontsize=8)

    # Panel 4: Fit + eval runtime comparison
    ax = axes[1, 1]
    ax.plot(df_cmp['M'], df_cmp['fit_eval_linear_s'], 's-', label='Fit + eval Linear RP')
    sub_t = df_cmp.dropna(subset=['fit_eval_relu_s'])
    ax.plot(sub_t['M'], sub_t['fit_eval_relu_s'], 'o--', label='Fit + eval ReLU RP')
    ax.axvline(FEATURE_DIM, linestyle=':', color='tab:red', alpha=0.8)
    ax.set_ylabel('Thời gian 10 task (giây)')
    ax.set_title('4. Chi phí tính toán tăng theo M ở cả hai biến thể')
    ax.legend(fontsize=8)

    for ax in axes.flat:
        ax.set_xscale('log')
        ax.set_xticks(LINEAR_SWEEP_DIMS, labels=[str(d) for d in LINEAR_SWEEP_DIMS], rotation=25)
        ax.minorticks_off()
        ax.set_xlabel('Kích thước projection M')
        ax.grid(alpha=0.25)

    with atomic_output(out_dir / 'linear_vs_relu_4panel.png') as tmp:
        fig.savefig(tmp, format='png', dpi=150)
    plt.show()
    print(f'Đã lưu bảng và hình đối chứng tại: {out_dir}')
